# Build a Causal Language Model Using PyTorch
In notebook07 we learned how to build a causal language model (GPT) from scratch. In this homework, we will build a Causal Language model with PyTorch, which also supports back-propagation compared to the notebook implementation.

In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
batch_size = 16 # how many independent sequences will we process in parallel?
block_size = 32 # what is the maximum context length for predictions?
max_iters = 1000
eval_interval = 100
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 64
n_head = 4
n_layer = 4
dropout = 0.0
# ------------

We are going to train our GPT on this corpus

In [2]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print(text[:1000])

--2026-06-17 10:15:48--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.04s   

2026-06-17 10:15:49 (28.9 MB/s) - ‘input.txt’ saved [1115394/1115394]

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it b

For simplicity, character-level tokenization is applied here instead of learning a new BPE tokenizer

In [3]:
# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y


### TODO1: Single-head Attention Implementation

In [4]:
class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        # register_buffer simply adds self.tril as a non-paramter buffer in this nn.Module
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # TODO: fill in the forward pass for single-head attention
        # Hint 1: use self.tril as the attention mask
        # Hint 2: use torch.tensor.masked_fill to mask out the attention scores
        # Hint 3: See the MultiHeadAttention for how the heads are concatenated

        # ===== my answer below =====
        # x: (B, T, C) where C = n_embd
        B, T, C = x.shape
        k = self.key(x)    # project to keys:    (B, T, head_size)
        q = self.query(x)  # project to queries: (B, T, head_size)
        v = self.value(x)  # project to values:  (B, T, head_size)

        # compute scaled attention scores ("affinities") between every query and key
        # (B, T, hs) @ (B, hs, T) -> (B, T, T); divide by sqrt(head_size) to keep softmax sharp
        head_size = k.shape[-1]
        wei = q @ k.transpose(-2, -1) * head_size ** -0.5

        # Hint 1 & 2: use the lower-triangular tril buffer as a causal mask so a token
        # can only attend to itself and earlier tokens. Slice to (T, T) so it also works
        # when T < block_size (e.g. during generation). masked_fill sets future scores to -inf.
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)  # normalize scores into weights: (B, T, T)
        wei = self.dropout(wei)       # regularization

        # weighted aggregation of the values
        out = wei @ v  # (B, T, T) @ (B, T, hs) -> (B, T, hs)
        return out
        # ===== end of my answer =====

### Multi-head attention and feed forward layer implementations

In [5]:
class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

### TODO2: Transformer Block Implementation

In [6]:
class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        # TODO: fill in the forward pass
        # Hint: don't forget the residual connections

        # ===== my answer below =====
        # pre-norm formulation (LayerNorm before each sub-layer), as used in GPT.
        # the "x +" parts are the residual (skip) connections.
        x = x + self.sa(self.ln1(x))    # communication: multi-head self-attention
        x = x + self.ffwd(self.ln2(x))  # computation: position-wise feed-forward
        return x
        # ===== end of my answer =====

### Language Model Implementation.

In [7]:
class LanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

In [8]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

model = LanguageModel()
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))

0.209729 M parameters
step 0: train loss 4.3293, val loss 4.3227
step 100: train loss 2.6536, val loss 2.6586
step 200: train loss 2.5046, val loss 2.5052
step 300: train loss 2.3923, val loss 2.4042
step 400: train loss 2.3178, val loss 2.3310
step 500: train loss 2.2532, val loss 2.2724
step 600: train loss 2.2094, val loss 2.2413
step 700: train loss 2.1679, val loss 2.1911
step 800: train loss 2.1234, val loss 2.1505
step 900: train loss 2.0901, val loss 2.1316
step 999: train loss 2.0586, val loss 2.0961

That my grese powea ced sed Ceed!uce,
Thance that theeis noss his lodirdss:
Rompas in bouly by babeeitonenale Soy roffnis lewill marre,
That dose sor barket stering hums.
Niph Rabrolices your hwith rate? as I dream thanen.

TMENENNER, Howhaild your shay, and chean.
Nursass, it thous hame bose kinat:
Talings; that thisurdougnoinke, Sreech favow thoum neart
But anne to bestentake thy you
lickes spor ands of wan sOm' a all fall;
Gee willttant suence. stidit, my in goluguran gaed to


### TODO3: BERTopic Practice
Use BERTopic to obtain top-5 topics in AG's news dataset

In [12]:
# Install BERTopic (and a compatible sentence-transformers backend).
# On Colab this takes a minute or two; restart the runtime if prompted.
!pip install -q bertopic sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 5.7 MB/s eta 0:00:00


In [9]:
!wget https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/train.csv

--2026-06-17 10:25:31--  https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/train.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 29470338 (28M) [text/plain]
Saving to: ‘train.csv’

train.csv           100%[===================>]  28.10M  --.-KB/s    in 0.1s    

2026-06-17 10:25:32 (206 MB/s) - ‘train.csv’ saved [29470338/29470338]



In [10]:
import pandas as pd
df = pd.read_csv("train.csv", header=None)

# Analyze the topic distribution of first 5000 news leading sentences
docs = df.iloc[:, 2].tolist()
docs = docs[:5000]

In [13]:
# TODO: conduct topic modeling following the steps of the last section in
# https://github.com/elliottash/lm_lss_2024/blob/main/notebooks/08_LLMs.ipynb

# ===== my answer below =====
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

# 1. Embed the documents with a sentence-transformer.
#    all-MiniLM-L6-v2 is small and runs comfortably on a T4 GPU.
embedding_model = SentenceTransformer("all-MiniLM-L6-v2", device=device)
embeddings = embedding_model.encode(docs, show_progress_bar=True)

# 2. Fit BERTopic. We pass the precomputed embeddings so the embedding step
#    isn't repeated. (BERTopic then does UMAP -> HDBSCAN -> c-TF-IDF internally.)
topic_model = BERTopic(embedding_model=embedding_model, verbose=True)
topics, probs = topic_model.fit_transform(docs, embeddings)

# 3. Inspect the topics. Topic -1 is the "outlier" topic, so the real topics
#    are the rows with Topic >= 0, ordered by frequency.
topic_info = topic_model.get_topic_info()
print(topic_info.head(10))
# ===== end of my answer =====

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

2026-06-17 10:30:27,524 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-06-17 10:31:08,005 - BERTopic - Dimensionality - Completed ✓
2026-06-17 10:31:08,008 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-06-17 10:31:08,310 - BERTopic - Cluster - Completed ✓
2026-06-17 10:31:08,334 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-06-17 10:31:08,797 - BERTopic - Representation - Completed ✓


   Topic  Count                                 Name  \
0     -1   1224                     -1_of_the_and_in   
1      0    333               0_night_season_run_his   
2      1    272     1_google_offering_initial_public   
3      2    107            2_oil_prices_crude_barrel   
4      3     91     3_sharon_ariel_israeli_jerusalem   
5      4     90       4_athens_olympia_games_ancient   
6      5     82  5_drugs_drug_healthdaynews_patients   
7      6     64        6_championship_golf_vijay_pga   
8      7     63      7_basketball_team_states_greece   
9      8     61      8_music_realnetworks_apple_ipod   

                                      Representation  \
0     [of, the, and, in, to, on, for, its, that, is]   
1  [night, season, run, his, league, inning, ap, ...   
2  [google, offering, initial, public, search, ip...   
3  [oil, prices, crude, barrel, record, demand, 4...   
4  [sharon, ariel, israeli, jerusalem, israel, pr...   
5  [athens, olympia, games, ancient, olympics, 

In [14]:
# Show the top-5 topics (excluding the -1 outlier topic) with their keywords.
top5 = [t for t in topic_info["Topic"].tolist() if t != -1][:5]
for t in top5:
    keywords = ", ".join(w for w, _ in topic_model.get_topic(t))
    count = int(topic_info.loc[topic_info["Topic"] == t, "Count"].iloc[0])
    print(f"Topic {t} (n={count}): {keywords}")

# Optional: a bar chart of the topic keywords (renders inline in Colab).
topic_model.visualize_barchart(top_n_topics=5)

Topic 0 (n=333): night, season, run, his, league, inning, ap, game, sox, the
Topic 1 (n=272): google, offering, initial, public, search, ipo, stock, inc, shares, price
Topic 2 (n=107): oil, prices, crude, barrel, record, demand, 47, fresh, london, high
Topic 3 (n=91): sharon, ariel, israeli, jerusalem, israel, prime, minister, party, west, palestinian
Topic 4 (n=90): athens, olympia, games, ancient, olympics, olympic, greece, shot, competition, quot
